# Sales Lead Data Cleaning

This notebook turns the messy raw export (`data/leads_raw.csv`) into the clean, analysis-ready dataset used everywhere else in this project (`data/leads_cleaned.csv` — the file the SQL load step reads into MySQL).

**Data quality issues fixed here, in order:**
1. Duplicate `lead_id` rows (a sync job re-exported some leads)
2. Inconsistent `source` / `product` text (casing, abbreviations, stray spaces)
3. Negative `response_time_hours` (a sign-flip entry error)
4. Mixed date formats on `created_at`
5. `deal_value` stored as a currency string (`"$480.00"`) for some rows
6. Missing `region` values, imputed from each rep's most common region


In [1]:
import pandas as pd

raw = pd.read_csv("../data/leads_raw.csv")
print(f"Raw rows: {len(raw)}")
raw.head()


Raw rows: 7105


,lead_id,created_at,source,product,region,rep,response_time_hours,stage_reached,status,lost_reason,deal_value,closed_at
0,LD-300000,11/14/2025 14:30,Webinar,Coding for Kids,NaN,Rep_15,0.57,Closed,Won,NaN,574.72,2025-12-15 09:41:11.226466
1,LD-300001,2026-06-12 16:52:00,Organic Search,Language Learning,NaN,Rep_01,2.72,Contacted,Open,NaN,NaN,NaN
2,LD-300002,21-Jan-2026 15:02,Organic Search,Coding for Kids,South Region,Rep_12,1.63,Demo Completed,Lost,Went With Competitor,673.32,2026-02-25 19:23:42.793161
3,LD-300003,28-Aug-2025 15:26,Referral,K-12 Math,South Region,Rep_07,1.77,Contacted,Open,NaN,NaN,NaN
4,LD-300004,06/03/2026 10:00,Paid Search,Language Learning,West Region,Rep_13,2.73,Lead,Open,NaN,NaN,NaN


## Step 1: Understand what's wrong with the raw data

Before fixing anything, measure the damage - never clean blind.


In [2]:
print("=== Data quality report: raw export ===")
print(f"Duplicate lead_id rows:        {raw['lead_id'].duplicated().sum()}")
print(f"Distinct source strings:       {raw['source'].nunique()} (should be 7)")
print(sorted(raw['source'].unique()))
print(f"Distinct product strings:      {raw['product'].nunique()} (should be 5)")
print(sorted(raw['product'].unique()))
print(f"Negative response_time_hours:  {(pd.to_numeric(raw['response_time_hours'], errors='coerce') < 0).sum()}")
print(f"Missing region:                {raw['region'].isna().sum()}")
print(f"deal_value dtype sample:       {raw['deal_value'].dropna().head(3).tolist()}")
print(f"created_at format sample:      {raw['created_at'].head(3).tolist()}")


=== Data quality report: raw export ===
Duplicate lead_id rows:        105
Distinct source strings:       19 (should be 7)
['COLD OUTREACH', 'Cold Outreach', 'Cold-Outreach', 'Organic Search', 'Organic search', 'PPC', 'Paid  Search', 'Paid Search', 'Partner Channel', 'Referral', 'SEO', 'SOCIAL MEDIA', 'Social', 'Social Media', 'Webinar', 'cold outreach', 'organic search', 'paid search', 'social media']
Distinct product strings:      11 (should be 5)
['CODING FOR KIDS', 'Coding For Kids', 'Coding for Kids', 'Competitive Exam Prep', 'K-12  Math', 'K-12 Math', 'K12 Math', 'Language Learning', 'Study Abroad Prep', 'coding for kids', 'k-12 math']
Negative response_time_hours:  12
Missing region:                426
deal_value dtype sample:       ['574.72', '673.32', '723.67']
created_at format sample:      ['11/14/2025 14:30', '2026-06-12 16:52:00', '21-Jan-2026 15:02']


## Step 2: Remove duplicate lead exports

The same `lead_id` appearing more than once means the same lead got exported twice by a sync job, not two different leads. Keep the first occurrence, drop the rest.


In [3]:
df = raw.drop_duplicates(subset="lead_id", keep="first").reset_index(drop=True)
print(f"Rows after dedup: {len(df)} (removed {len(raw) - len(df)})")


Rows after dedup: 7000 (removed 105)


## Step 3: Standardize `source` and `product` text

Both columns have the same handful of real categories typed inconsistently across an 18-month export history - different casing, extra whitespace, and outright abbreviations (`PPC`, `SEO`). A lookup table maps every variant actually observed in this export back to one canonical spelling. This is safer than a generic `.str.lower().str.strip()` pass because abbreviations like `PPC` aren't just a casing problem - they need an explicit mapping.


In [4]:
SOURCE_MAP = {
    "organic search": "Organic Search", "Organic search": "Organic Search", "SEO": "Organic Search",
    "paid search": "Paid Search", "PPC": "Paid Search", "Paid  Search": "Paid Search",
    "social media": "Social Media", "Social": "Social Media", "SOCIAL MEDIA": "Social Media",
    "cold outreach": "Cold Outreach", "Cold-Outreach": "Cold Outreach", "COLD OUTREACH": "Cold Outreach",
}
PRODUCT_MAP = {
    "k-12 math": "K-12 Math", "K12 Math": "K-12 Math", "K-12  Math": "K-12 Math",
    "coding for kids": "Coding for Kids", "Coding For Kids": "Coding for Kids", "CODING FOR KIDS": "Coding for Kids",
}

n_source_fixed = df["source"].isin(SOURCE_MAP).sum()
n_product_fixed = df["product"].isin(PRODUCT_MAP).sum()
df["source"] = df["source"].replace(SOURCE_MAP)
df["product"] = df["product"].replace(PRODUCT_MAP)
print(f"Standardized {n_source_fixed} source values -> {df['source'].nunique()} distinct sources")
print(f"Standardized {n_product_fixed} product values -> {df['product'].nunique()} distinct products")


Standardized 692 source values -> 7 distinct sources
Standardized 529 product values -> 5 distinct products


## Step 4: Fix impossible negative response times

A response time can't be negative - this is a sign-flip data-entry error, not a real measurement, so the fix is to take the absolute value rather than drop the rows (the magnitude is still trustworthy, just the sign is wrong).


In [5]:
df["response_time_hours"] = pd.to_numeric(df["response_time_hours"], errors="coerce")
n_negative = (df["response_time_hours"] < 0).sum()
df["response_time_hours"] = df["response_time_hours"].abs()
print(f"Fixed {n_negative} rows with a negative response_time_hours")


Fixed 12 rows with a negative response_time_hours


## Step 5: Parse the mixed date formats on `created_at`

The export mixes three date formats, most likely because the export format changed partway through the 18-month period. Try each known format in turn rather than trusting pandas to guess - silent misparsing (e.g. swapping day/month) is worse than an explicit failure.


In [6]:
def parse_mixed_date(value):
    for fmt in ("%m/%d/%Y %H:%M", "%d-%b-%Y %H:%M", "%Y-%m-%d %H:%M:%S"):
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            continue
    return pd.to_datetime(value)  # last resort - let pandas infer


df["created_at"] = df["created_at"].apply(parse_mixed_date)
print(f"created_at dtype: {df['created_at'].dtype}")
print(f"range: {df['created_at'].min()} to {df['created_at'].max()}")


created_at dtype: datetime64[ns]
range: 2025-01-01 08:15:00 to 2026-06-29 20:24:00


## Step 6: Convert `deal_value` currency strings to numbers

A subset of rows have `deal_value` stored as text like `"$480.00"` instead of a plain number - can't do any math on a string. Strip the `$` and thousands separator, then convert to numeric.


In [7]:
n_currency_strings = df["deal_value"].apply(lambda v: isinstance(v, str) and "$" in v).sum()
df["deal_value"] = pd.to_numeric(
    df["deal_value"].astype(str).str.replace(r"[$,]", "", regex=True),
    errors="coerce",
)
print(f"Converted {n_currency_strings} currency-formatted strings to numbers")
print(f"deal_value dtype now: {df['deal_value'].dtype}")


Converted 730 currency-formatted strings to numbers
deal_value dtype now: float64


## Step 7: Fill missing `region` from each rep's most common region

~6% of rows have no `region`. Rather than dropping those rows or guessing blindly, use a signal that's actually reliable here: every rep is consistently assigned to one home region, so the rep's own most common region is a strong stand-in for a missing value on their lead.


In [8]:
n_missing_region = df["region"].isna().sum()
rep_home_region = df.dropna(subset=["region"]).groupby("rep")["region"].agg(lambda s: s.mode().iloc[0])

df["region"] = df.apply(
    lambda row: rep_home_region.get(row["rep"], row["region"]) if pd.isna(row["region"]) else row["region"],
    axis=1,
)
print(f"Filled {n_missing_region} missing region values using each rep's most common region")
print(f"Remaining missing region: {df['region'].isna().sum()}")


Filled 418 missing region values using each rep's most common region
Remaining missing region: 0


## Step 8: Final validation

Check the cleaning actually worked before saving anything - never trust, always verify.


In [9]:
assert df["lead_id"].duplicated().sum() == 0, "no duplicate lead_id should remain"
assert df["source"].nunique() == 7, f"expected 7 canonical sources, got {df['source'].nunique()}"
assert df["product"].nunique() == 5, f"expected 5 canonical products, got {df['product'].nunique()}"
assert (df["response_time_hours"] < 0).sum() == 0, "no negative response times should remain"
assert df["created_at"].dtype.kind == "M", "created_at should be a proper datetime now"
assert df["deal_value"].dtype.kind == "f", "deal_value should be fully numeric now"
assert df["region"].isna().sum() == 0, "no missing region should remain"
print("All checks passed.")


All checks passed.


## Step 9: Save the cleaned dataset


In [10]:
df.to_csv("../data/leads_cleaned.csv", index=False)
print(f"Saved {len(df)} rows to data/leads_cleaned.csv")


Saved 7000 rows to data/leads_cleaned.csv
